## 1. Setup

In [1]:
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/tokenizerv3.pkl .
# !mkdir ./base-models
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/gqa-rope.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/mla-sin.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/gqa-rope-swiglu-moe.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/swiglu-moe-rope-conv.pt ./base-models/

In [2]:
import os, gc, copy, time, math, random, re, json, pickle, zipfile, hashlib
from dataclasses import dataclass, replace
from collections import defaultdict, Counter

import numpy as np, pandas as pd
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score

import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Data location / field names (check against the EDA cell output) ──
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE5_DIR      = "Phase_5_LSI"
ID_KEY, TEXT_KEY, LABELS_KEY = "id", "text", "labels"

# True -> tiny subsets + separate output dir, to verify the whole pipeline in minutes
SMOKE_TEST = False
OUT = "./cs781_phase5_smoke" if SMOKE_TEST else "./cs781_phase5"
os.makedirs(OUT, exist_ok=True)

# ── Phase 1 model hyper-parameters (LMConfig reads these) ──
VOCAB_SIZE       = 20_000
TRAIN_BLOCK_SIZE = 2048
N_EMBD           = 768
N_LAYER          = 12
N_HEAD           = 12
N_KV_HEAD        = 4
KV_LATENT_DIM    = N_EMBD // 4

PHASE1_CONFIGS = {
    "gqa-rope":             {"n_layers": 8,  "d_ff": 4 * 768, "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": False, "use_conv": False},
    "mla-sin":              {"n_layers": 12, "d_ff": 4 * 768, "attention_type": "mla", "pos_emb_type": "sinusoidal", "use_moe": False, "use_conv": False},
    "gqa-rope-swiglu-moe":  {"n_layers": 12, "d_ff": 1408,    "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": True,  "use_conv": False},
    "swiglu-moe-rope-conv": {"n_layers": 12, "d_ff": 1408,    "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": True,  "use_conv": True, "local_kernel_size": 25},
}
MAIN_MODEL = "gqa-rope-swiglu-moe"   # model used for the technique / ablation experiments

# ── Long-document geometry ──
CHUNK_LEN     = 512                  # tokens per chunk, including the <s> token
CHUNK_OVERLAP = 64                   # tokens shared by consecutive chunks
CONTENT_LEN   = CHUNK_LEN - 1        # real tokens per chunk
CHUNK_STRIDE  = CONTENT_LEN - CHUNK_OVERLAP
DEFAULT_MAX_CHUNKS = 8               # chunks per case fed to the model (memory/speed knob)

# ── Training ──
BATCH_DOCS      = 4          # cases per step
GRAD_ACCUM      = 4          # effective batch = BATCH_DOCS * GRAD_ACCUM
EVAL_BATCH_DOCS = 8
GRAD_CKPT       = True       # recompute activations in backward -> much lower memory
WARMUP_FRAC     = 0.06
WEIGHT_DECAY    = 0.01
HEAD_DROPOUT    = 0.1
FOCAL_GAMMA     = 2.0
POS_WEIGHT_CAP  = 30.0       # upper bound of the per-statute positive weight
REPEAT_THRESHOLD = 0.02      # repeat-factor sampling: statutes in <2% of cases are oversampled
ADALORA_ORTH_COEF = 0.01

# ── Thresholding ──
THRESHOLD_GRID = np.arange(0.02, 0.90, 0.02)
PER_LABEL_MIN_SUPPORT = 5    # a statute needs >= this many dev positives to get its own threshold
THRESHOLD_MODE = "global"    # "global" | "per_label": which thresholds the submission uses

SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 2. Tokenizer

In [3]:
class ByteLevelBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.reserved = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos = list(self.reserved); self.stoi = {}; self.merges = []; self.ranks = {}; self.merged_map = {}; self._cache = {}
        # Regex pattern matching words, numbers, individual punctuation, and whitespace sequences
        self.pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        self.pat = re.compile(self.pattern, re.UNICODE)

    def fit(self, texts, max_unique_words=200000):
        """Train a byte-level BPE tokenizer using inverted indexing for fast merging."""
        raw_counts = Counter()
        iterator = [texts] if isinstance(texts, str) else texts

        # Preserve whitespace tokens when populating initial raw_counts
        for text in tqdm(iterator, desc="Building word vocabulary"):
            for chunk in re.findall(self.pattern, str(text), flags=re.UNICODE):
                raw_counts[chunk] += 1

        if max_unique_words and len(raw_counts) > max_unique_words:
            most_common = raw_counts.most_common(max_unique_words)
            raw_counts = dict(most_common)

        word_freqs = {}; pair_counts = defaultdict(int); pair_to_words = defaultdict(set)
        for word, count in raw_counts.items():
            encoded = word.encode("utf-8")
            byte_word = tuple(encoded[i:i+1] for i in range(len(encoded)))
            word_freqs[byte_word] = count

            for i in range(len(byte_word) - 1):
                pair = (byte_word[i], byte_word[i + 1])
                pair_counts[pair] += count
                pair_to_words[pair].add(byte_word)

        base_bytes = sorted({token for word in word_freqs for token in word})
        self.itos = list(self.reserved) + base_bytes; self.merges = []
        target_merges = self.vocab_size - len(self.itos)

        with tqdm(total=target_merges, desc="Training BPE Merges") as pbar:
            while len(self.itos) < self.vocab_size:
                if not pair_counts:
                    break
                best_pair = max(pair_counts, key=lambda p: (pair_counts[p], p[0], p[1]))
                if pair_counts[best_pair] <= 0:
                    break
                merged_token = best_pair[0] + best_pair[1]
                self.merges.append((best_pair, merged_token))
                self.itos.append(merged_token)
                
                words_to_update = list(pair_to_words[best_pair])
                del pair_counts[best_pair]
                del pair_to_words[best_pair]

                p0, p1 = best_pair
                for word in words_to_update:
                    count = word_freqs.get(word, 0)
                    if count == 0:
                        continue

                    new_word = []; i = 0; n = len(word)
                    while i < n:
                        if i + 1 < n and word[i] == p0 and word[i + 1] == p1:
                            new_word.append(merged_token)
                            i += 2
                        else:
                            new_word.append(word[i])
                            i += 1
                    new_word = tuple(new_word)
                    
                    for j in range(len(word) - 1):
                        old_p = (word[j], word[j + 1])
                        if old_p in pair_counts:
                            pair_counts[old_p] -= count
                            if pair_counts[old_p] <= 0:
                                del pair_counts[old_p]

                    for j in range(len(new_word) - 1):
                        new_p = (new_word[j], new_word[j + 1])
                        pair_counts[new_p] += count
                        pair_to_words[new_p].add(new_word)

                    del word_freqs[word]
                    word_freqs[new_word] = count

                pbar.update(1)

        self.vocab_size = len(self.itos)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}
        self.ranks = {pair: rank for rank, (pair, _) in enumerate(self.merges)}
        self.merged_map = {pair: merged for pair, merged in self.merges}
        self._cache = {}
        return self

    def encode_word(self, word):
        """Encode single word using memoization and local pair updates."""
        if word in self._cache:
            return self._cache[word]

        encoded = word.encode("utf-8")
        tokens = [encoded[i:i+1] for i in range(len(encoded))]

        if len(tokens) <= 1:
            res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
            self._cache[word] = res
            return res

        while len(tokens) > 1:
            min_rank = float("inf")
            min_pair = None

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self.ranks.get(pair, float("inf"))
                if rank < min_rank:
                    min_rank = rank
                    min_pair = pair

            if min_pair is None or min_rank == float("inf"):
                break

            merged = self.merged_map[min_pair]
            new_tokens = []
            i = 0
            n = len(tokens)
            p0, p1 = min_pair

            while i < n:
                if i + 1 < n and tokens[i] == p0 and tokens[i + 1] == p1:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
        self._cache[word] = res
        return res

    def encode(self, text):
        """Encode a sentence while preserving original spaces and newlines."""
        ids = [self.stoi["<s>"]]
        pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        for chunk in re.findall(pattern, str(text), flags=re.UNICODE):
            ids.extend(self.encode_word(chunk))
        ids.append(self.stoi["</s>"])
        return ids
    
    def decode(self, ids):
        """Decode BPE token IDs back into text seamlessly."""
        byte_list = []
        for idx in ids:
            tok = self.itos[idx] if 0 <= idx < len(self.itos) else "<unk>"
            if tok not in self.reserved and isinstance(tok, bytes):
                byte_list.append(tok)
        try:
            return b"".join(byte_list).decode("utf-8", errors="ignore")
        except Exception:
            return ""

    def get_vocab_size(self):
        return self.vocab_size

def train_tokenizer(train_text: str, vocab_size: int):
    """Train A-1 BPE tokenizer on `train_text` and return the trained tokenizer."""
    tokenizer = ByteLevelBPE(vocab_size)
    tokenizer.fit(train_text)
    return tokenizer
    
# Load Pretrained Tokenizer
with open("tokenizerv3.pkl", "rb") as f:
    tokenizer = pickle.load(f)
BOS, PAD, UNK, EOS = (tokenizer.stoi[t] for t in ("<s>", "<pad>", "<unk>", "</s>"))
assert len(tokenizer.itos) == VOCAB_SIZE
print("tokenizer vocab:", len(tokenizer.itos))

tokenizer vocab: 20000


## 3. Model Class Definitions

In [4]:
@dataclass
class LMConfig:
    vocab_size: int = VOCAB_SIZE
    d_model: int = N_EMBD
    n_layers: int = N_LAYER
    n_heads: int = N_HEAD
    n_kv_heads: int = N_KV_HEAD
    kv_latent_dim: int = KV_LATENT_DIM
    d_ff: int = 1408          # For SwiGLU (2/3) * 4 * d_model => 2/3 * 2048 = 1365 --> round to nearest upper multiple of 64 (1408)
    max_seq_len: int  = TRAIN_BLOCK_SIZE
    dropout: float = 0.0            # ~1 pass over the data -> no need for dropout
    pos_emb_type: str = "rope"      # "rope" or "sinusoidal"
    attention_scale: float = None
    norm_first: bool = True         # FIX: pre-LN (post-LN + lr 1e-3 + no warm-up is unstable)
    n_experts:  int   = 4
    top_k:      int   = 2
    aux_loss_coef: float = 0.01
    attention_type: str = "gqa"
    use_moe: bool = False, 
    use_conv: bool = False
    local_kernel_size: int = 25
    
# Rotary Positional Embeddings
def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """ Returns (cos, sin) : the split-half rotate_half convention: channel i is paired with channel i+d_head/2. """
    inv = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    ang = torch.cat([torch.outer(torch.arange(max_seq_len).float(), inv)] * 2, -1)
    return ang.cos(), ang.sin()
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree rotation step used by the "rotate_half" formulation of RoPE."""
    x1, x2 = x[ ..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 : ]
    return torch.cat((-x2, x1), dim = -1)
def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """ x: (B, n_heads, T, d_head) -- a query or key tensor. cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T. Returns the rotated tensor, same shape as x. Formula: x_rotated = x * cos + rotate_half(x) * sin """
    cos, sin = (cos[None, None], sin[None, None]) if cos.dim() == 2 else (cos[:, None], sin[:, None])
    return x.float() * cos + rotate_half(x.float()) * sin

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    i = torch.arange(0, d_model, 2, dtype=torch.float32)
    angle = pos / (10000 ** (i / d_model))
    return torch.stack([torch.sin(angle), torch.cos(angle)], dim=-1).reshape(max_len, d_model)

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=True, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep, self.d_head = n_heads // n_kv_heads, d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids=None, attn_mask=None, kv_cache=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[position_ids], self.rope_sin[position_ids])
            k = apply_rope(k, self.rope_cos[position_ids], self.rope_sin[position_ids])
             
        # Append to KV cache if provided
        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)
        new_kv_cache = (k, v) if use_cache else None

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # Only set is_causal=True if processing multiple prompt tokens at once (T > 1)
        is_causal = (attn_mask is None) and (kv_cache is None) and (T > 1)
        context = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_dropout.p if self.training else 0.0,  is_causal=is_causal, scale=self.attention_scale)
        out = self.resid_dropout(self.out_proj(context.transpose(1, 2).reshape(B, T, C)))
        
        return (out, new_kv_cache) if use_cache else out

class MultiHeadLatentAttention(nn.Module):
    """Joint KV Compression Multi-Head Latent Attention module drop-in replacement for CausalSelfAttention."""

    def __init__(self, d_model, n_heads, kv_latent_dim, max_seq_len, dropout=0.0, attention_scale=None):
        super().__init__()
        assert (d_model % n_heads == 0 and 0 < kv_latent_dim < d_model), "d_model must be divisible by n_heads, and 0 < kv_latent_dim < d_model"

        self.n_heads, self.kv_latent_dim, self.d_head = n_heads, kv_latent_dim, d_model // n_heads
        self.attention_scale = ((1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale))

        # Projections
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_down_proj = nn.Linear(d_model, kv_latent_dim, bias=False)
        self.k_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.v_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        # Dropouts
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, position_ids=None, attn_mask=None, kv_cache=None, use_cache=False):
        """Forward pass supporting sequence generation with KV cache and FlashAttention/SDPA."""
        B, T, C = x.shape
        q = (self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T, d_head)
        c_kv = self.kv_down_proj(x)  # (B, T, kv_latent_dim)

        if kv_cache is not None:
            past_c_kv = (kv_cache[0] if isinstance(kv_cache, (tuple, list)) else kv_cache)
            c_kv = torch.cat([past_c_kv, c_kv], dim=1)  # Append along sequence dim T
        new_kv_cache = (c_kv,) if use_cache else None

        k = (self.k_up_proj(c_kv).view(B, -1, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T_total, d_head)
        v = (self.v_up_proj(c_kv).view(B, -1, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T_total, d_head)
        is_causal = (attn_mask is None) and (kv_cache is None) and (T > 1)
        
        context = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=is_causal, scale=self.attention_scale)
        out = context.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))

        return (out, new_kv_cache) if use_cache else out

### Model Definitions
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)
        
class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False); self.up   = nn.Linear(d_model, d_ff, bias=False); self.down = nn.Linear(d_ff, d_model, bias=False); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class TopKRouter(nn.Module):
    def __init__(self, d_model: int, n_experts: int, top_k: int):
        super().__init__()
        self.top_k  = top_k; self.linear = nn.Linear(d_model, n_experts, bias=False)
    def forward(self, x_flat):                           
        logits = self.linear(x_flat); topk_vals, topk_idx = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(topk_vals, dim=-1)            # (B*T, top_k)
        dispatch = torch.zeros_like(logits, dtype=weights.dtype)
        dispatch.scatter_(-1, topk_idx, weights)          # (B*T, n_experts)
        return dispatch, logits                           # logits kept for aux loss

class MoEFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, n_experts, top_k, dropout, aux_coef=0.01):
        super().__init__()
        self.router   = TopKRouter(d_model, n_experts, top_k)
        self.experts  = nn.ModuleList([SwiGLUFeedForward(d_model, d_ff, dropout) for _ in range(n_experts)])
        self.aux_coef = aux_coef; self.aux_loss = torch.tensor(0.0)

    def forward(self, x):
        B, T, D = x.shape
        x_flat = x.reshape(B * T, D)
        dispatch, router_logits = self.router(x_flat)

        z = torch.logsumexp(router_logits, dim=-1)
        self.aux_loss = self.aux_coef * (z ** 2).mean()
        out = torch.zeros_like(x_flat)
        for e_idx, expert in enumerate(self.experts):
            w  = dispatch[:, e_idx];  mask = w > 0
            if not mask.any():
                continue
            out[mask] += w[mask].unsqueeze(-1) * expert(x_flat[mask])

        return out.reshape(B, T, D)

class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.norm_first = cfg.norm_first; self.ln1 = nn.LayerNorm(cfg.d_model); self.ln2 = nn.LayerNorm(cfg.d_model)
        if(cfg.attention_type == "mla"):
            self.attn = MultiHeadLatentAttention(cfg.d_model, cfg.n_heads, cfg.kv_latent_dim, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        else:
            self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"), attention_scale=cfg.attention_scale)
        if(cfg.use_moe):
            self.ffn  = MoEFeedForward(cfg.d_model, cfg.d_ff, cfg.n_experts, cfg.top_k, cfg.dropout, cfg.aux_loss_coef)
        else:
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        
    def _attn(self, h, position_ids, attn_mask, kv_cache, use_cache):
        r = self.attn(h, position_ids=position_ids, attn_mask=attn_mask, kv_cache=kv_cache, use_cache=use_cache)
        return r if use_cache else (r, None)
        
    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        if self.norm_first:
            a, new_cache = self._attn(self.ln1(x), position_ids, attn_mask, kv_cache, use_cache)
            x = x + a
            x = x + self.ffn(self.ln2(x))
        else:
            a, new_cache = self._attn(x, position_ids, attn_mask, kv_cache, use_cache)
            x = self.ln1(x + a)
            x = self.ln2(x + self.ffn(x))
        return x, new_cache
        
class LocalDepthwiseConv(nn.Module):
    """ Depthwise causal convolution over the sequence dimension. Acts on each channel independently — O(T * C * kernel) cost. kernel_size = 25 covers ~25 tokens of local context. """
    def __init__(self, d_model: int, kernel_size: int = 25, dropout: float = 0.0):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=kernel_size, padding=0, groups=d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        nn.init.normal_(self.conv.weight, 0.0, 0.02)
        nn.init.normal_(self.proj.weight, 0.0, 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, C) → (B, T, C)"""
        B, T, C = x.shape
        xc = x.transpose(1, 2)
        xc = F.pad(xc, (self.kernel_size - 1, 0))
        xc = self.conv(xc)          # (B, C, T)  — output length = T (causal)
        xc = xc.transpose(1, 2)    # (B, T, C)
        return self.drop(self.proj(F.gelu(xc)))

class ParallelLocalGlobalBlock(nn.Module):
    """Pre-LN block with global attention and local conv running IN PARALLEL."""
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.RMSNorm(cfg.d_model); self.ln2  = nn.RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, use_rope=cfg.pos_emb_type == "rope", attention_scale=cfg.attention_scale)
        self.conv = LocalDepthwiseConv(cfg.d_model, kernel_size=cfg.local_kernel_size, dropout=cfg.dropout)
        self.ffn  = MoEFeedForward(cfg.d_model, cfg.d_ff, cfg.n_experts, cfg.top_k, cfg.dropout, cfg.aux_loss_coef)
        self.alpha = nn.Parameter(torch.ones(1))    # global attention scale
        self.beta  = nn.Parameter(torch.full((1,), 0.1))  # local conv scale

    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        normed = self.ln1(x)
        attn_out = self.attn(normed, position_ids=position_ids, attn_mask=attn_mask, kv_cache=kv_cache, use_cache=use_cache)
        attn_out, new_cache = attn_out if use_cache else attn_out, None
        conv_out = self.conv(normed)
        
        x = x + self.alpha * attn_out + self.beta * conv_out
        x = x + self.ffn(self.ln2(x))
        return x, new_cache

    @property
    def aux_loss(self):
        return self.ffn.aux_loss


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.use_sin = cfg.pos_emb_type == "sinusoidal"
        self.emb_scale = math.sqrt(cfg.d_model) if self.use_sin else 1.0
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if self.use_sin:
            self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model), persistent=False)
        if cfg.use_conv:
            self.blocks = nn.ModuleList([ParallelLocalGlobalBlock(cfg) for _ in range(cfg.n_layers)])
        else:
            self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self._init_weights()
        self.head.weight = self.tok_emb.weight # tie AFTER init so the embedding init survives

    def aux_loss(self):
        return sum(blk.ffn.aux_loss for blk in self.blocks)
        
    def _init_weights(self):
        emb_std = self.cfg.d_model ** -0.5 if self.use_sin else 0.02
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=emb_std)
        for n, p in self.named_parameters():
            if n.endswith("attn.out_proj.weight") or n.endswith("ffn.net.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * self.cfg.n_layers))
    
    def forward(self, idx: torch.Tensor, kv_cache_list=None, use_cache=False, attention_mask=None) -> torch.Tensor:
        """attention_mask: optional (B, past+T) tensor, 1 = real token, 0 = left padding."""
        B, T = idx.shape
        past = kv_cache_list[0][0].shape[-2] if kv_cache_list is not None else 0
        S = past + T

        if attention_mask is None:
            position_ids = torch.arange(past, S, device=idx.device).unsqueeze(0).expand(B, T)
            attn_mask = None                                   # fast path: is_causal=True inside SDPA
        else:
            position_ids = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, past:S]
            q_idx = torch.arange(past, S, device=idx.device)[:, None]
            k_idx = torch.arange(S, device=idx.device)[None, :]
            allowed = (k_idx <= q_idx)[None] & attention_mask[:, None, :S].bool()   # causal & not-padding
            allowed = allowed | (k_idx == q_idx)[None]         # pad rows attend to themselves (no NaNs)
            attn_mask = allowed.unsqueeze(1)                   # (B, 1, T, S)

        x = self.tok_emb(idx) * self.emb_scale
        if self.use_sin:
            x = x + self.pos_emb[position_ids]

        new_caches = []
        for i, block in enumerate(self.blocks):
            layer_cache = kv_cache_list[i] if kv_cache_list is not None else None
            x, c = block(x, position_ids, attn_mask, layer_cache, use_cache)
            new_caches.append(c)

        logits = self.head(self.ln_f(x))
        return (logits, new_caches) if use_cache else logits
        
    @torch.no_grad()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=384, temperature=0.7, top_p=0.9,
                 repetition_penalty=1.0, banned_token_ids=(0, 1, 2, 3), do_sample=True):
        self.eval()
        out = input_ids
        if attention_mask is None:
            attention_mask = torch.ones_like(out)
    
        logits, kv = self(out, use_cache=True, attention_mask=attention_mask)      # prefill
        next_logits = logits[:, -1, :].float()
    
        for step in range(max_new_tokens):
            l = next_logits.clone()
            if repetition_penalty != 1.0:
                sc = l.gather(1, out); sc = torch.where(sc < 0, sc * repetition_penalty, sc / repetition_penalty)
                l = l.scatter(1, out, sc)
            if banned_token_ids:                       # never emit <pad>/<unk>/<s>/</s>: we need 128 words
                l[:, list(banned_token_ids)] = float("-inf")

            if do_sample and temperature > 0.0:
                l = l / temperature
                if top_p < 1.0:
                    sorted_logits, sorted_idx = torch.sort(l, descending=True)
                    cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    remove = cum > top_p
                    remove[..., 1:] = remove[..., :-1].clone()
                    remove[..., 0] = False
                    l = l.masked_fill(remove.scatter(1, sorted_idx, remove), float("-inf"))
                next_token = torch.multinomial(F.softmax(l, dim=-1), num_samples=1)
            else:
                next_token = l.argmax(dim=-1, keepdim=True)

            out = torch.cat([out, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)
            if step == max_new_tokens - 1:
                break
            logits, kv = self(next_token, kv_cache_list=kv, use_cache=True, attention_mask=attention_mask)
            next_logits = logits[:, -1, :].float()
        return out

## 4. Data Loading

In [5]:
def load_phase1_model(model_name):
    cfg = LMConfig()
    for attr, val in PHASE1_CONFIGS[model_name].items():
        setattr(cfg, attr, val)

    checkpoint_data = torch.load(f"./base-models/{model_name}.pt", map_location="cpu", weights_only=False)
    base_lm = DecoderLM(cfg)
    base_lm.load_state_dict(checkpoint_data["model"], strict=True)
    del checkpoint_data; gc.collect()
    return base_lm

@dataclass
class CaseSplit:
    ids: list                    # case ids, in file order
    texts: list                  # cleaned case text
    fingerprints: list           # md5 of cleaned text (dedup + token cache key)
    label_lists: list            # sorted statute ids per case (None for test)
    labels: np.ndarray           # multi-hot [N, NUM_LABELS] uint8 (None for test; filled below)
    token_ids: list              # filled by the tokenization cell

def clean_text(text):
    """Light cleanup. Keep consistent with whatever was applied to the Phase 1 pretraining text."""
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)          # collapse runs of spaces/tabs
    text = re.sub(r"\n{3,}", "\n\n", text)       # collapse many blank lines
    return text.strip()

def fingerprint(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def read_split(relative_path):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset", filename=f"{PHASE5_DIR}/{relative_path}")
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)
    assert isinstance(records, list), f"{relative_path}: expected a list of records, got {type(records)}"
    return records

def build_split(records, has_labels, split_name):
    ids = [r[ID_KEY] for r in records]
    assert len(set(ids)) == len(ids), f"{split_name}: duplicate case ids"
    assert all(isinstance(r[TEXT_KEY], str) for r in records), f"{split_name}: non-string text"
    texts = [clean_text(r[TEXT_KEY]) for r in records]
    assert all(len(t) > 0 for t in texts), f"{split_name}: empty text after cleaning"
    label_lists = None
    if has_labels:
        for r in records:
            labs = r[LABELS_KEY]
            assert isinstance(labs, list) and len(labs) > 0, f"{split_name}: case {r[ID_KEY]} has no labels"
            assert all(isinstance(l, int) and not isinstance(l, bool) and l >= 0 for l in labs), f"{split_name}: case {r[ID_KEY]} has non-int / negative label"
            # assert len(set(labs)) == len(labs), f"{split_name}: case {r[ID_KEY]} repeats a label"
        label_lists = [sorted(r[LABELS_KEY]) for r in records]
    return CaseSplit(ids, texts, [fingerprint(t) for t in texts], label_lists, None, None)

def select_rows(split, rows):
    rows = list(rows)
    return CaseSplit(
        ids=[split.ids[i] for i in rows],
        texts=[split.texts[i] for i in rows],
        fingerprints=[split.fingerprints[i] for i in rows],
        label_lists=None if split.label_lists is None else [split.label_lists[i] for i in rows],
        labels=None if split.labels is None else split.labels[rows],
        token_ids=None if split.token_ids is None else [split.token_ids[i] for i in rows],
    )

train_split = build_split(read_split("Train/train.json"), True,  "train")
dev_split   = build_split(read_split("Train/dev.json"),   True,  "dev")
test_split  = build_split(read_split("Test/test.json"),   False, "test")
print("raw sizes -> train", len(train_split.ids), "| dev", len(dev_split.ids), "| test", len(test_split.ids))

# 1) identical texts inside train: keep the first row and MERGE (union) the label sets; report disagreements
first_row, keep_rows, label_conflicts = {}, [], 0
merged_labels = [list(ls) for ls in train_split.label_lists]
for row, fp in enumerate(train_split.fingerprints):
    if fp in first_row:
        keeper = first_row[fp]
        label_conflicts += int(set(merged_labels[keeper]) != set(train_split.label_lists[row]))
        merged_labels[keeper] = sorted(set(merged_labels[keeper]) | set(train_split.label_lists[row]))
    else:
        first_row[fp] = row; keep_rows.append(row)
train_split.label_lists = merged_labels
print(f"train: removed {len(train_split.ids) - len(keep_rows)} duplicate texts ({label_conflicts} with different label sets, merged)")
train_split = select_rows(train_split, keep_rows)

# 2) dev cases whose text also appears in train (or repeats inside dev) would inflate model selection -> drop them
train_fps, seen_in_dev, dev_keep = set(train_split.fingerprints), set(), []
n_in_train = n_repeat = 0
for i, fp in enumerate(dev_split.fingerprints):
    if fp in train_fps:
        n_in_train += 1
    elif fp in seen_in_dev:
        n_repeat += 1
    else:
        seen_in_dev.add(fp); dev_keep.append(i)
print(f"dev: removed {n_in_train} cases that also occur in train and {n_repeat} repeated within dev")
dev_split = select_rows(dev_split, dev_keep)

# 3) test is never filtered (one prediction per test id is required); only report overlap
print("test texts also present in train:", sum(fp in train_fps for fp in test_split.fingerprints))
print("id overlap train/dev:", len(set(train_split.ids) & set(dev_split.ids)),
      "| train/test:", len(set(train_split.ids) & set(test_split.ids)))

if SMOKE_TEST:
    rng = np.random.RandomState(SEED)
    train_split = select_rows(train_split, rng.choice(len(train_split.ids), 64, replace=False))
    dev_split   = select_rows(dev_split,   rng.choice(len(dev_split.ids),   32, replace=False))
    test_split  = select_rows(test_split,  rng.choice(len(test_split.ids),   32, replace=False))

# ── label space: statute ids seen in train or dev, mapped to contiguous columns ──
LABEL_IDS    = sorted({l for ls in train_split.label_lists + dev_split.label_lists for l in ls})
LABEL_TO_COL = {label_id: col for col, label_id in enumerate(LABEL_IDS)}
NUM_LABELS   = len(LABEL_IDS)

def to_multi_hot(label_lists):
    matrix = np.zeros((len(label_lists), NUM_LABELS), dtype=np.uint8)
    for row, ls in enumerate(label_lists):
        matrix[row, [LABEL_TO_COL[l] for l in ls]] = 1
    return matrix

train_split.labels = to_multi_hot(train_split.label_lists)
dev_split.labels   = to_multi_hot(dev_split.label_lists)

TRAIN_LABEL_COUNTS = train_split.labels.sum(0)                 # positives per statute in train
PREDICTABLE        = TRAIN_LABEL_COUNTS > 0                    # only these statutes may be predicted
TRAIN_LABEL_PRIOR  = torch.tensor((TRAIN_LABEL_COUNTS + 1) / (len(train_split.ids) + 2), dtype=torch.float32)  # Laplace-smoothed
print(f"label space: {NUM_LABELS} statutes (id range {LABEL_IDS[0]}..{LABEL_IDS[-1]}), {int(PREDICTABLE.sum())} have train examples")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

raw sizes -> train 42750 | dev 10181 | test 13019
train: removed 435 duplicate texts (208 with different label sets, merged)
dev: removed 118 cases that also occur in train and 26 repeated within dev
test texts also present in train: 150
id overlap train/dev: 0 | train/test: 0
label space: 100 statutes (id range 0..99), 100 have train examples


In [6]:
TOKEN_CACHE_PATH = f"{OUT}/token_cache_tokenizerv3_{VOCAB_SIZE}.pkl"
token_cache = pickle.load(open(TOKEN_CACHE_PATH, "rb")) if os.path.exists(TOKEN_CACHE_PATH) else {}

def tokenize_split(split, split_name):
    """Token ids of the whole case WITHOUT <s>/</s> (each chunk gets its own <s>)."""
    for text, fp in tqdm(zip(split.texts, split.fingerprints), total=len(split.texts), desc=f"tokenize {split_name}"):
        if fp not in token_cache:
            token_cache[fp] = tokenizer.encode(text)[1:-1]
    split.token_ids = [token_cache[fp] for fp in split.fingerprints]

for name, split in [("train", train_split), ("dev", dev_split), ("test", test_split)]:
    tokenize_split(split, name)
pickle.dump(token_cache, open(TOKEN_CACHE_PATH, "wb"))

# sanity: byte-level BPE should almost never emit <unk>
all_tokens = np.concatenate([np.array(t) for t in train_split.token_ids])
print("unk fraction in train tokens:", (all_tokens == UNK).mean())

def chunk_starts(num_tokens):
    """Start offsets of overlapping windows; the last window is aligned to the end of the document."""
    if num_tokens <= CONTENT_LEN:
        return [0]
    starts = list(range(0, num_tokens - CONTENT_LEN + 1, CHUNK_STRIDE))
    if starts[-1] + CONTENT_LEN < num_tokens:
        starts.append(num_tokens - CONTENT_LEN)
    return starts

REGION_THIRDS = {"first_third": 0, "middle_third": 1, "last_third": 2}   # used for the section probe

def pick_evenly(candidates, k):
    if len(candidates) <= k:
        return candidates
    positions = np.unique(np.linspace(0, len(candidates) - 1, k).round().astype(int))
    return [candidates[p] for p in positions]

def select_chunk_indices(num_chunks, strategy, max_chunks):
    """Which chunks of a case are fed to the model."""
    all_idx = list(range(num_chunks))
    if strategy == "head":
        return all_idx[:max_chunks]
    if strategy == "tail":
        return all_idx[-max_chunks:]
    if strategy == "head_tail":                                  # beginning + end of the case
        if num_chunks <= max_chunks:
            return all_idx
        n_head = max_chunks // 2
        return all_idx[:n_head] + all_idx[-(max_chunks - n_head):]
    if strategy == "uniform":                                    # evenly spaced over the whole case
        return pick_evenly(all_idx, max_chunks)
    if strategy in REGION_THIRDS:                                # only one third of the case
        third = REGION_THIRDS[strategy]
        lo = num_chunks * third // 3
        hi = max(num_chunks * (third + 1) // 3, lo + 1)
        return pick_evenly(all_idx[lo:hi], max_chunks)
    raise ValueError(f"unknown chunk strategy: {strategy}")

class CaseDataset(Dataset):
    """One item = one case = a few chunks (+ multi-hot labels, + optional teacher logits)."""
    def __init__(self, split, strategy, max_chunks):
        self.split, self.strategy, self.max_chunks = split, strategy, max_chunks
        self.teacher_logits = None                              # set for the KD student

    def __len__(self):
        return len(self.split.ids)

    def __getitem__(self, i):
        token_ids = self.split.token_ids[i]
        starts = chunk_starts(len(token_ids))
        chosen = select_chunk_indices(len(starts), self.strategy, self.max_chunks)
        item = {"chunks": [[BOS] + token_ids[starts[c]: starts[c] + CONTENT_LEN] for c in chosen]}
        if self.split.labels is not None:
            item["labels"] = torch.from_numpy(self.split.labels[i].astype(np.float32))
        if self.teacher_logits is not None:
            item["teacher_logits"] = self.teacher_logits[i]
        return item

def collate_cases(items):
    """Pad to ids [B, max_chunks_in_batch, max_len] and chunk_mask [B, max_chunks_in_batch]."""
    n_chunks = max(len(it["chunks"]) for it in items)
    max_len  = max(len(c) for it in items for c in it["chunks"])
    ids = torch.full((len(items), n_chunks, max_len), PAD, dtype=torch.long)
    chunk_mask = torch.zeros(len(items), n_chunks, dtype=torch.bool)
    for b, it in enumerate(items):
        for c, chunk in enumerate(it["chunks"]):
            ids[b, c, :len(chunk)] = torch.tensor(chunk)
            chunk_mask[b, c] = True
    batch = {"ids": ids, "chunk_mask": chunk_mask}
    if "labels" in items[0]:
        batch["labels"] = torch.stack([it["labels"] for it in items])
    if "teacher_logits" in items[0]:
        batch["teacher_logits"] = torch.stack([it["teacher_logits"] for it in items])
    return batch

tokenize test: 100%|██████████| 13019/13019 [00:00<00:00, 895850.05it/s]


unk fraction in train tokens: 9.487293653906811e-05


In [7]:
n_train = len(train_split.ids)
print(f"train n={n_train} | dev n={len(dev_split.ids)} | test n={len(test_split.ids)}")

# ── label statistics ──
labels_per_case = train_split.labels.sum(1)
print("labels/case percentiles (min/25/50/75/90/99/max):", np.percentile(labels_per_case, [0, 25, 50, 75, 90, 99, 100]).astype(int))
print(f"label density (positives / cells): {train_split.labels.mean():.4f}")
for low, high in [(1, 4), (5, 19), (20, 99), (100, 10**9)]:
    in_bucket = (TRAIN_LABEL_COUNTS >= low) & (TRAIN_LABEL_COUNTS <= high)
    print(f"statutes with {low}-{high if high < 10**9 else 'inf'} train cases: {int(in_bucket.sum())}")
top = np.argsort(-TRAIN_LABEL_COUNTS)[:10]
print("most frequent statute ids:", [(LABEL_IDS[c], int(TRAIN_LABEL_COUNTS[c])) for c in top])

dev_support = dev_split.labels.sum(0)
unseen_in_dev = (dev_support > 0) & ~PREDICTABLE
print(f"dev statutes never seen in train: {int(unseen_in_dev.sum())} (unpredictable, they will score 0 F1)")
print("dev cases whose labels are ALL unseen in train:", int(((dev_split.labels * PREDICTABLE).sum(1) == 0).sum()))

cooc = train_split.labels.T.astype(np.int32) @ train_split.labels.astype(np.int32)
print("distinct co-occurring statute pairs in train:", int((np.triu(cooc, 1) > 0).sum()))

# ── length statistics (note: the default MAX_CHUNKS may truncate long cases) ──
train_lengths = np.array([len(t) for t in train_split.token_ids])
train_chunks  = np.array([len(chunk_starts(n)) for n in train_lengths])
print("tokens/case percentiles (50/75/90/95/99/max):", np.percentile(train_lengths, [50, 75, 90, 95, 99, 100]).astype(int))
print("chunks/case percentiles (50/75/90/95/99/max):", np.percentile(train_chunks,  [50, 75, 90, 95, 99, 100]).astype(int))
for k in (4, 8, 16):
    print(f"cases that fit entirely in {k} chunks: {(train_chunks <= k).mean():.1%}")
print("correlation(tokens, #labels):", np.corrcoef(train_lengths, labels_per_case)[0, 1].round(3))

# ── eyeball a few cases ──
for i in range(3):
    print(f"\n--- id {train_split.ids[i]} | statute ids {train_split.label_lists[i]} | last 300 chars ---\n{train_split.texts[i][-300:]}")

train n=42315 | dev n=10037 | test n=13019
labels/case percentiles (min/25/50/75/90/99/max): [ 1  2  3  5  7 12 37]
label density (positives / cells): 0.0371
statutes with 1-4 train cases: 0
statutes with 5-19 train cases: 0
statutes with 20-99 train cases: 0
statutes with 100-inf train cases: 100
most frequent statute ids: [(5, 11577), (34, 10304), (97, 7817), (43, 7665), (39, 7056), (12, 4288), (13, 4100), (10, 4077), (77, 3990), (94, 3820)]
dev statutes never seen in train: 0 (unpredictable, they will score 0 F1)
dev cases whose labels are ALL unseen in train: 0
distinct co-occurring statute pairs in train: 4707
tokens/case percentiles (50/75/90/95/99/max): [   657   2106   4252   6115  12600 144663]
chunks/case percentiles (50/75/90/95/99/max): [  2   5  10  14  29 324]
cases that fit entirely in 4 chunks: 72.3%
cases that fit entirely in 8 chunks: 87.1%
cases that fit entirely in 16 chunks: 96.5%
correlation(tokens, #labels): 0.146

--- id 1000008 | statute ids [3, 9, 69] | last 3

## 5. Finetuning methods

In [8]:
class LoRALinear(nn.Module):
    """y = Wx + (alpha/r) * B(A x)   (additive low-rank update)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """y = Wx + (alpha/r) * B(A(Wx))  ==  (I + s*BA) W x   (multiplicative low-rank update)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)

class QLoRALinear(nn.Module):
    """Frozen weight stored as int8 (per-output-channel absmax quantisation, dequantised on the fly) + trainable LoRA.
    Weight-only 8-bit simulation of QLoRA (no bitsandbytes needed)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        assert base.bias is None, "QLoRALinear expects bias-free projections"
        weight = base.weight.detach()
        scale = weight.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / 127.0
        self.register_buffer("w_int8", torch.round(weight / scale).clamp(-127, 127).to(torch.int8))
        self.register_buffer("w_scale", scale)
        self.lora_scale = alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        dequantised = self.w_int8.float() * self.w_scale
        return F.linear(x, dequantised) + self.lora_scale * F.linear(F.linear(x, self.A), self.B)

class AdaLoRALinear(nn.Module):
    """Delta W = P diag(lambda * mask) Q. Orthogonality penalty + rank pruning (see AdaLoRAController)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.rank, self.scale = base, rank, alpha / rank
        self.P    = nn.Parameter(torch.empty(base.out_features, rank))
        self.diag = nn.Parameter(torch.zeros(rank))
        self.Q    = nn.Parameter(torch.empty(rank, base.in_features))
        self.register_buffer("mask", torch.ones(rank))           # 0 = pruned singular value
        nn.init.orthogonal_(self.P); nn.init.orthogonal_(self.Q)
    def orth_loss(self):
        eye = torch.eye(self.rank, device=self.P.device)
        return ((self.P.T @ self.P - eye) ** 2).sum() + ((self.Q @ self.Q.T - eye) ** 2).sum()
    def forward(self, x):
        low_rank = F.linear(x, self.Q) * (self.diag * self.mask)
        return self.base(x) + self.scale * F.linear(low_rank, self.P)

class AdaLoRAController:
    """Prunes singular values by smoothed sensitivity |lambda * grad| following a cubic budget schedule."""
    def __init__(self, modules, total_steps, final_ratio, warmup_frac=0.1, freeze_frac=0.2, prune_every=20, beta=0.85):
        self.modules, self.prune_every, self.beta = modules, prune_every, beta
        self.initial_budget = sum(m.rank for m in modules)
        self.final_budget = int(self.initial_budget * final_ratio)
        self.start_step = int(total_steps * warmup_frac)
        self.end_step = int(total_steps * (1 - freeze_frac))
        self.importance = [torch.zeros_like(m.diag) for m in modules]
    def budget_at(self, step):
        if step < self.start_step: return self.initial_budget
        if step >= self.end_step:  return self.final_budget
        progress = (step - self.start_step) / (self.end_step - self.start_step)
        return int(self.final_budget + (self.initial_budget - self.final_budget) * (1 - progress) ** 3)
    @torch.no_grad()
    def update(self, step):
        """Call after backward (grads unscaled), before optimizer.step()."""
        if step >= self.end_step:
            return
        for module, imp in zip(self.modules, self.importance):
            imp.mul_(self.beta).add_((1 - self.beta) * (module.diag * module.diag.grad).abs())
        if step >= self.start_step and step % self.prune_every == 0:
            all_scores = torch.cat(self.importance)
            threshold = torch.topk(all_scores, self.budget_at(step)).values[-1]
            for module, imp in zip(self.modules, self.importance):
                module.mask.copy_((imp >= threshold).float())
    def active_rank(self):
        return int(sum(m.mask.sum().item() for m in self.modules))

class AdapterBlock(nn.Module):
    """Houlsby-style bottleneck adapter applied to the output of a transformer block."""
    def __init__(self, block, d_model, bottleneck):
        super().__init__()
        self.block = block
        self.down, self.up = nn.Linear(d_model, bottleneck), nn.Linear(bottleneck, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        h, cache = self.block(x, position_ids, attn_mask)
        return h + self.up(F.gelu(self.down(h))), cache

class SoftPrefix(nn.Module):
    """n learned vectors prepended to the embeddings of every chunk (backbone stays frozen).
    reparam=False -> prompt tuning; reparam=True -> prefix-style MLP reparameterisation of the vectors."""
    def __init__(self, n_tokens, dim, token_embeddings, reparam):
        super().__init__()
        init_rows = torch.randint(0, token_embeddings.size(0), (n_tokens,))
        self.latent = nn.Parameter(token_embeddings[init_rows].clone())
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.Tanh(), nn.Linear(dim, dim)) if reparam else None
    def forward(self):
        return self.latent if self.mlp is None else self.mlp(self.latent)

PEFT_WRAPPERS = {"lora": LoRALinear, "lorma": LoRMALinear, "qlora": QLoRALinear, "adalora": AdaLoRALinear}
ATTN_PROJ_NAMES = ("q_proj", "k_proj", "v_proj", "out_proj", "kv_down_proj", "k_up_proj", "v_up_proj")  # GQA and MLA names

def wrap_attention_projections(backbone, wrapper_cls, rank, alpha):
    """Wrap every attention projection that exists in this architecture (GQA / MLA / conv variants)."""
    for block in backbone.blocks:
        for name in ATTN_PROJ_NAMES:
            if hasattr(block.attn, name):
                setattr(block.attn, name, wrapper_cls(getattr(block.attn, name), rank, alpha))

In [9]:
def encode_chunks(backbone, ids, use_ckpt, prefix_emb):
    """Phase 1 decoder over [N, L] chunks (causal, right-padded) -> hidden states [N, L, D].
    Right padding + causal attention means padding never influences real tokens, so no mask is needed.
    Optional soft-prefix vectors [n, D] are prepended to every chunk and removed from the output."""
    N, L = ids.shape
    x = backbone.tok_emb(ids) * backbone.emb_scale
    n_prefix = 0
    if prefix_emb is not None:
        n_prefix = prefix_emb.size(0)
        x = torch.cat([prefix_emb.unsqueeze(0).expand(N, -1, -1), x], dim=1)
    total_len = n_prefix + L
    position_ids = torch.arange(total_len, device=ids.device).unsqueeze(0).expand(N, total_len)
    if backbone.use_sin:
        x = x + backbone.pos_emb[position_ids]
    for block in backbone.blocks:
        if use_ckpt and torch.is_grad_enabled():
            x = checkpoint(lambda h, blk=block: blk(h, position_ids, None)[0], x, use_reentrant=False)
        else:
            x = block(x, position_ids, None)[0]
    return backbone.ln_f(x)[:, n_prefix:]

# ── pooling of chunk vectors [B, C, D] into one case vector [B, D] ──
class MeanPooler(nn.Module):
    def forward(self, grid, mask):
        m = mask.unsqueeze(-1).to(grid.dtype)
        return (grid * m).sum(1) / m.sum(1)

class AttentionPooler(nn.Module):
    """Learned attention over chunks."""
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)
    def forward(self, grid, mask):
        scores = self.score(grid).squeeze(-1).float().masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)
        return (weights.unsqueeze(-1) * grid.float()).sum(1)

class BiGRUAttentionPooler(nn.Module):
    """Hierarchical: BiGRU over the ordered chunk sequence (section-aware), then attention pooling."""
    def __init__(self, dim):
        super().__init__()
        self.gru = nn.GRU(dim, dim // 2, batch_first=True, bidirectional=True)
        self.attn = AttentionPooler(dim)
    def forward(self, grid, mask):
        packed = pack_padded_sequence(grid.float(), mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=grid.size(1))
        return self.attn(grid.float() + out, mask)

# ── multi-label heads: (chunk grid [B, C, D], chunk_mask [B, C]) -> logits [B, NUM_LABELS]. `out` is the output layer ──
class PooledHead(nn.Module):
    """Pool chunks into one case vector, then one linear layer (binary relevance)."""
    def __init__(self, pooler, dim, num_labels):
        super().__init__()
        self.pooler, self.dropout, self.out = pooler, nn.Dropout(HEAD_DROPOUT), nn.Linear(dim, num_labels)
    def forward(self, grid, mask):
        return self.out(self.dropout(self.pooler(grid, mask)))

class ChunkMaxHead(nn.Module):
    """Chunk-level statute logits, then max over chunks: a statute is present if ANY chunk shows it."""
    def __init__(self, dim, num_labels):
        super().__init__()
        self.dropout, self.out = nn.Dropout(HEAD_DROPOUT), nn.Linear(dim, num_labels)
    def forward(self, grid, mask):
        chunk_logits = self.out(self.dropout(grid)).float().masked_fill(~mask.unsqueeze(-1), float("-inf"))
        return chunk_logits.max(dim=1).values

class LabelAttentionHead(nn.Module):
    """Every statute has its own attention over chunks: logit_l = sum_c softmax_c(a_l(h_c)) * s_l(h_c)."""
    def __init__(self, dim, num_labels):
        super().__init__()
        self.dropout, self.attn, self.out = nn.Dropout(HEAD_DROPOUT), nn.Linear(dim, num_labels), nn.Linear(dim, num_labels)
    def forward(self, grid, mask):
        grid = self.dropout(grid)
        attn_logits = self.attn(grid).float().masked_fill(~mask.unsqueeze(-1), float("-inf"))   # [B, C, L]
        weights = torch.softmax(attn_logits, dim=1)
        return (weights * self.out(grid).float()).sum(dim=1)

HEADS = {"mean":       lambda d, n: PooledHead(MeanPooler(), d, n),
         "attn":       lambda d, n: PooledHead(AttentionPooler(d), d, n),
         "bigru":      lambda d, n: PooledHead(BiGRUAttentionPooler(d), d, n),
         "chunk_max":  ChunkMaxHead,
         "label_attn": LabelAttentionHead}

def init_output_layer(layer, label_prior):
    """Weights ~ N(0, 0.02); bias = log-odds of the label prior so training starts at the base rates."""
    nn.init.normal_(layer.weight, std=0.02)
    with torch.no_grad():
        layer.bias.copy_(torch.logit(label_prior))

class CaseClassifier(nn.Module):
    def __init__(self, backbone, head_type, use_ckpt, label_prior, soft_prefix):
        super().__init__()
        self.backbone, self.use_ckpt, self.soft_prefix = backbone, use_ckpt, soft_prefix
        self.head = HEADS[head_type](backbone.cfg.d_model, NUM_LABELS)
        init_output_layer(self.head.out, label_prior)

    def forward(self, ids, chunk_mask):
        B, C, L = ids.shape
        flat_ids = ids[chunk_mask]                                   # only real chunks: [N, L]
        prefix_emb = self.soft_prefix() if self.soft_prefix is not None else None
        hidden = encode_chunks(self.backbone, flat_ids, self.use_ckpt, prefix_emb)
        token_w = flat_ids.ne(PAD).unsqueeze(-1).to(hidden.dtype)
        chunk_repr = (hidden * token_w).sum(1) / token_w.sum(1).clamp(min=1.0)       # masked mean over tokens
        grid = chunk_repr.new_zeros(B, C, chunk_repr.size(-1))
        grid[chunk_mask] = chunk_repr                                # back to [B, C, D]
        return self.head(grid, chunk_mask)

# ── model factory ──
def freeze(module):
    for p in module.parameters():
        p.requires_grad = False

def build_model(base_lm, exp):
    backbone = copy.deepcopy(base_lm)
    if exp.block_stride > 1:                                         # KD student: keep every n-th block
        keep = list(range(0, len(backbone.blocks), exp.block_stride))
        backbone.blocks = nn.ModuleList([backbone.blocks[i] for i in keep])
    d_model = backbone.cfg.d_model
    soft_prefix = None

    if exp.method == "full":
        pass                                                         # everything trainable
    elif exp.method == "partial":                                    # top-n blocks + final norm
        freeze(backbone)
        for block in backbone.blocks[-exp.n_top:]:
            for p in block.parameters(): p.requires_grad = True
        for p in backbone.ln_f.parameters(): p.requires_grad = True
    elif exp.method in PEFT_WRAPPERS:
        freeze(backbone)
        wrap_attention_projections(backbone, PEFT_WRAPPERS[exp.method], exp.rank, exp.alpha)
    elif exp.method == "adapter":
        freeze(backbone)
        backbone.blocks = nn.ModuleList([AdapterBlock(b, d_model, exp.adapter_dim) for b in backbone.blocks])
    elif exp.method in ("prompt", "prefix"):
        freeze(backbone)
        soft_prefix = SoftPrefix(exp.n_prefix, d_model, backbone.tok_emb.weight.detach(), reparam=(exp.method == "prefix"))
    else:
        raise ValueError(f"unknown method: {exp.method}")
    return CaseClassifier(backbone, exp.head_type, GRAD_CKPT, TRAIN_LABEL_PRIOR, soft_prefix).to(DEVICE)

def count_params(model):
    """(total, trainable) in millions. Quantised int8 weights (QLoRA) are counted as parameters."""
    total = sum(p.numel() for p in model.parameters()) + sum(b.numel() for n, b in model.named_buffers() if n.endswith("w_int8"))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total / 1e6, trainable / 1e6

def model_size_mb(model):
    """Real storage of weights (int8 weights of QLoRA count 1 byte)."""
    n_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    n_bytes += sum(b.numel() * b.element_size() for n, b in model.named_buffers() if n.endswith(("w_int8", "w_scale")))
    return n_bytes / 2**20

## 7. Training Utilities

In [10]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

# ── imbalance handling ──
def positive_class_weights(labels):
    """Per-statute positive weight sqrt(neg/pos), clipped to [1, POS_WEIGHT_CAP]."""
    pos = labels.sum(0).astype(np.float64)
    neg = len(labels) - pos
    weights = np.clip(np.sqrt(neg / np.maximum(pos, 1)), 1.0, POS_WEIGHT_CAP)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def repeat_factor_weights(labels):
    """Per-case sampling weight: the rarest statute of the case decides how often the case is repeated."""
    freq = labels.mean(0)
    label_repeat = np.where(freq > 0, np.maximum(1.0, np.sqrt(REPEAT_THRESHOLD / np.maximum(freq, 1e-12))), 1.0)
    return (labels * label_repeat[None, :]).max(axis=1)

def compute_loss(logits, targets, pos_weight, loss_type):
    """Multi-label loss: every statute is an independent binary decision."""
    if loss_type == "bce":                                           # no imbalance handling
        return F.binary_cross_entropy_with_logits(logits, targets)
    if loss_type == "weighted_bce":                                  # up-weights positives of rare statutes
        return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)
    if loss_type == "focal":                                         # down-weights easy cells + positive weights
        weighted = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction="none")
        pt = torch.exp(-F.binary_cross_entropy_with_logits(logits, targets, reduction="none"))
        return (((1 - pt) ** FOCAL_GAMMA) * weighted).mean()
    raise ValueError(loss_type)

# ── decisions: probabilities -> statute sets ──
def decide(probs, thresholds, predictable):
    """Boolean prediction matrix. Only statutes seen in train can be predicted; every case gets at least one statute."""
    pred = (probs >= thresholds[None, :]) & predictable[None, :]
    empty_rows = np.where(~pred.any(axis=1))[0]
    masked_probs = np.where(predictable[None, :], probs, -1.0)
    pred[empty_rows, masked_probs[empty_rows].argmax(axis=1)] = True
    return pred

def f1_by_threshold(probs, gold, grid):
    """Per-statute F1 for every threshold in grid -> array [len(grid), NUM_LABELS] (0 where undefined)."""
    gold_bool = gold.astype(bool)
    n_pos = gold_bool.sum(0)
    f1 = np.zeros((len(grid), gold.shape[1]))
    for g, threshold in enumerate(grid):
        pred = probs >= threshold
        tp = (pred & gold_bool).sum(0)
        fp = (pred & ~gold_bool).sum(0)
        fn = n_pos - tp
        denom = 2 * tp + fp + fn
        f1[g] = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    return f1

def tune_thresholds(probs, gold):
    """Returns (global threshold, per-statute thresholds). Statutes with too little support fall back to the global one."""
    f1 = f1_by_threshold(probs, gold, THRESHOLD_GRID)
    support = gold.sum(0)
    global_t = float(THRESHOLD_GRID[f1[:, support > 0].mean(axis=1).argmax()])
    per_label = np.full(gold.shape[1], global_t)
    enough = support >= PER_LABEL_MIN_SUPPORT
    per_label[enough] = THRESHOLD_GRID[f1[:, enough].argmax(axis=0)]
    return global_t, per_label

def multilabel_metrics(gold, pred):
    """Macro scores are over statutes that occur in `gold` (a constant rescaling of an evaluator that averages over all columns)."""
    gold, pred = gold.astype(int), pred.astype(int)
    active = gold.sum(0) > 0
    return {"F1_macro": float(f1_score(gold[:, active], pred[:, active], average="macro", zero_division=0)),
            "P_macro":  float(precision_score(gold[:, active], pred[:, active], average="macro", zero_division=0)),
            "R_macro":  float(recall_score(gold[:, active], pred[:, active], average="macro", zero_division=0)),
            "F1_micro": float(f1_score(gold, pred, average="micro", zero_division=0)),
            "subset_acc": float((gold == pred).all(axis=1).mean())}

def per_label_scores(gold, pred):
    """Per-statute (F1, precision, recall) arrays."""
    gold, pred = gold.astype(bool), pred.astype(bool)
    tp = (gold & pred).sum(0).astype(float); fp = (~gold & pred).sum(0); fn = (gold & ~pred).sum(0)
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    precision = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0)
    recall = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    return f1, precision, recall

def crossfit_predictions(probs, gold, mode):
    """Honest threshold estimate: tune on one half of dev, predict the other half, and vice versa."""
    fold = np.random.RandomState(SEED).permutation(len(gold)) % 2
    pred = np.zeros(gold.shape, dtype=bool)
    for k in (0, 1):
        tune_rows, apply_rows = fold != k, fold == k
        global_t, per_label_t = tune_thresholds(probs[tune_rows], gold[tune_rows])
        thresholds = per_label_t if mode == "per_label" else np.full(gold.shape[1], global_t)
        pred[apply_rows] = decide(probs[apply_rows], thresholds, PREDICTABLE)
    return pred

def fit_temperature(logits, gold):
    """Calibration: one global temperature minimising dev BCE."""
    grid = np.arange(0.5, 4.01, 0.1)
    targets = torch.tensor(gold, dtype=torch.float32)
    losses = [F.binary_cross_entropy_with_logits(torch.tensor(logits / t), targets).item() for t in grid]
    return float(grid[int(np.argmin(losses))])

def threshold_report(dev_probs, gold, global_t, per_label_t):
    """Dev metrics for every threshold strategy (in-sample = optimistic, cross-fit = honest)."""
    flat = np.full(NUM_LABELS, global_t)
    return {"fixed_0.5":          multilabel_metrics(gold, decide(dev_probs, np.full(NUM_LABELS, 0.5), PREDICTABLE)),
            "global_insample":    multilabel_metrics(gold, decide(dev_probs, flat, PREDICTABLE)),
            "per_label_insample": multilabel_metrics(gold, decide(dev_probs, per_label_t, PREDICTABLE)),
            "global_crossfit":    multilabel_metrics(gold, crossfit_predictions(dev_probs, gold, "global")),
            "per_label_crossfit": multilabel_metrics(gold, crossfit_predictions(dev_probs, gold, "per_label"))}

# ── model prediction ──
@torch.no_grad()
def predict_logits(model, dataset):
    model.eval()
    loader = DataLoader(dataset, batch_size=EVAL_BATCH_DOCS, shuffle=False, collate_fn=collate_cases)
    all_logits = []
    for batch in tqdm(loader, desc="predict", leave=False):
        with autocast("cuda", dtype=torch.float16):
            logits = model(batch["ids"].to(DEVICE), batch["chunk_mask"].to(DEVICE))
        all_logits.append(logits.float().cpu())
    return torch.cat(all_logits)

def evaluate_epoch(dev_logits, gold):
    """Per-epoch model selection: macro-F1 at the best global threshold."""
    probs = sigmoid(dev_logits)
    global_t, _ = tune_thresholds(probs, gold)
    metrics = multilabel_metrics(gold, decide(probs, np.full(NUM_LABELS, global_t), PREDICTABLE))
    return metrics, global_t

In [11]:
def checkpoint_path(name):
    return f"{OUT}/ckpt_{name}.pt"     # latest epoch (resume point), overwritten every epoch

def best_path(name):
    return f"{OUT}/best_{name}.pt"     # best epoch only, kept separately

def save_atomic(obj, path):
    """Write to a temp file, then rename: an interruption can never corrupt the previous file."""
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def get_rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all()}

def set_rng_state(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"]); torch.cuda.set_rng_state_all(state["cuda"])

def save_best(exp, best_state, best_metric, epoch):
    save_atomic({"state": best_state, "metric": best_metric, "epoch": epoch}, best_path(exp.name))

def save_epoch_checkpoint(exp, model, snapshot_keys, optimizer, scaler, adalora_ctrl, epoch, global_step, history):
    """Everything needed to continue training from the end of `epoch` (epochs completed so far)."""
    model_state = {k: v.detach().cpu() for k, v in model.state_dict().items() if k in snapshot_keys}
    save_atomic({"config": vars(exp), "epoch": epoch, "global_step": global_step, "history": history,
                 "model": model_state, "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                 "adalora_importance": None if adalora_ctrl is None else [imp.cpu() for imp in adalora_ctrl.importance],
                 "rng": get_rng_state()}, checkpoint_path(exp.name))

def resume_if_checkpointed(exp, model, optimizer, scaler, adalora_ctrl, fresh_history, worst_metric):
    """Returns (start_epoch, global_step, history, best_metric, best_state).
    Fresh start values are returned when this experiment has no checkpoint yet."""
    if not os.path.exists(checkpoint_path(exp.name)):
        return 0, 0, fresh_history, worst_metric, None

    ckpt = torch.load(checkpoint_path(exp.name), map_location="cpu", weights_only=False)
    # assert ckpt["config"] == vars(exp), f"{checkpoint_path(exp.name)} was written by a different configuration; delete it (and best_{exp.name}.pt) to start over"
    outcome = model.load_state_dict(ckpt["model"], strict=False)
    assert not outcome.unexpected_keys, outcome.unexpected_keys
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    if adalora_ctrl is not None:
        for importance, saved in zip(adalora_ctrl.importance, ckpt["adalora_importance"]):
            importance.copy_(saved)
    set_rng_state(ckpt["rng"])

    best = torch.load(best_path(exp.name), map_location="cpu", weights_only=False)
    print(f"[{exp.name}] resumed after epoch {ckpt['epoch']}/{exp.epochs} (best so far {best['metric']:.4f} at epoch {best['epoch']})")
    return ckpt["epoch"], ckpt["global_step"], ckpt["history"], best["metric"], best["state"]

## 8. Training Loop

In [12]:
def snapshot_names(model):
    """Parameters (and AdaLoRA masks) that must be restored to recover a given epoch."""
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    return trainable | {k for k in model.state_dict() if k.endswith(".mask")}

def make_loader(train_ds, exp):
    if exp.sampling == "rare_oversample":
        weights = torch.tensor(repeat_factor_weights(train_ds.split.labels), dtype=torch.double)
        sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
        return DataLoader(train_ds, batch_size=exp.batch_docs, sampler=sampler, collate_fn=collate_cases, pin_memory=True)
    assert exp.sampling == "random", exp.sampling
    return DataLoader(train_ds, batch_size=exp.batch_docs, shuffle=True, collate_fn=collate_cases, pin_memory=True)

def train_model(model, exp, train_ds, dev_ds):
    loader = make_loader(train_ds, exp)
    pos_weight = positive_class_weights(train_ds.split.labels)

    # backbone adaptation (adapters / LoRA / unfrozen layers / soft prefix) and the new head get different learning rates
    trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    is_body = lambda name: name.startswith("backbone.") or name.startswith("soft_prefix.")
    body_params = [p for n, p in trainable if is_body(n)]
    head_params = [p for n, p in trainable if not is_body(n)]
    optimizer = torch.optim.AdamW([{"params": body_params, "lr": exp.lr,      "weight_decay": WEIGHT_DECAY},
                                   {"params": head_params, "lr": exp.head_lr, "weight_decay": 0.0}])
    base_lrs = [exp.lr, exp.head_lr]

    steps_per_epoch = math.ceil(len(loader) / exp.grad_accum)
    total_steps = exp.epochs * steps_per_epoch
    warmup_steps = int(WARMUP_FRAC * total_steps)
    scaler = GradScaler("cuda")

    adalora_modules = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
    adalora_ctrl = AdaLoRAController(adalora_modules, total_steps, exp.adalora_final_ratio) if adalora_modules else None

    dev_labels = dev_ds.split.labels
    fresh_history = {"val_f1": [], "val_micro": [], "val_P": [], "val_R": [], "val_threshold": [], "loss": [], "train_sec": 0.0}
    start_epoch, global_step, history, best_f1, best_state = resume_if_checkpointed(
        exp, model, optimizer, scaler, adalora_ctrl, fresh_history, worst_metric=-1.0)

    for epoch in range(start_epoch, exp.epochs):
        t0 = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0

        for step_in_epoch, batch in enumerate(tqdm(loader, desc=f"{exp.name} ep {epoch + 1}/{exp.epochs}", leave=False)):
            ids, chunk_mask, targets = batch["ids"].to(DEVICE), batch["chunk_mask"].to(DEVICE), batch["labels"].to(DEVICE)
            with autocast("cuda", dtype=torch.float16):
                logits = model(ids, chunk_mask)
            logits = logits.float()

            loss = compute_loss(logits, targets, pos_weight, exp.loss_type)
            if "teacher_logits" in batch:                            # knowledge distillation: match the teacher's per-statute probabilities
                temperature = exp.kd_temperature
                soft_targets = torch.sigmoid(batch["teacher_logits"].to(DEVICE) / temperature)
                kd_loss = F.binary_cross_entropy_with_logits(logits / temperature, soft_targets) * temperature * temperature
                loss = exp.kd_hard_weight * loss + (1.0 - exp.kd_hard_weight) * kd_loss
            if adalora_modules:                                      # keep P, Q near-orthogonal
                loss = loss + ADALORA_ORTH_COEF * torch.stack([m.orth_loss() for m in adalora_modules]).mean()

            scaler.scale(loss / exp.grad_accum).backward()
            epoch_loss += loss.item()

            is_boundary = (step_in_epoch + 1) % exp.grad_accum == 0 or step_in_epoch == len(loader) - 1
            if is_boundary:
                # linear warm-up, then cosine decay
                if global_step < warmup_steps:
                    lr_frac = (global_step + 1) / max(1, warmup_steps)
                else:
                    progress = (global_step - warmup_steps) / max(1, total_steps - warmup_steps)
                    lr_frac = 0.5 * (1.0 + math.cos(math.pi * progress))
                for group, base_lr in zip(optimizer.param_groups, base_lrs):
                    group["lr"] = base_lr * lr_frac

                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                if adalora_ctrl is not None and torch.isfinite(grad_norm):
                    adalora_ctrl.update(global_step)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

        # ── validation: model selection by macro-F1 at the best global threshold ──
        dev_logits = predict_logits(model, dev_ds).numpy()
        metrics, val_threshold = evaluate_epoch(dev_logits, dev_labels)
        history["val_f1"].append(metrics["F1_macro"]); history["val_micro"].append(metrics["F1_micro"])
        history["val_P"].append(metrics["P_macro"]);   history["val_R"].append(metrics["R_macro"])
        history["val_threshold"].append(val_threshold)
        history["loss"].append(epoch_loss / len(loader)); history["train_sec"] += time.time() - t0
        extra = f" | active AdaLoRA rank {adalora_ctrl.active_rank()}/{adalora_ctrl.initial_budget}" if adalora_ctrl else ""
        print(f"[{exp.name}] ep {epoch + 1}/{exp.epochs} loss {history['loss'][-1]:.4f} | dev macro P {metrics['P_macro']:.3f} "
              f"R {metrics['R_macro']:.3f} F1 {metrics['F1_macro']:.4f} | micro-F1 {metrics['F1_micro']:.4f} @thr {val_threshold:.2f}{extra}")

        # ── checkpoints: best model (only when improved) first, then the resume point ──
        snapshot_keys = snapshot_names(model)
        if metrics["F1_macro"] > best_f1:
            best_f1 = metrics["F1_macro"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in snapshot_keys}
            save_best(exp, best_state, best_f1, epoch + 1)
        save_epoch_checkpoint(exp, model, snapshot_keys, optimizer, scaler, adalora_ctrl, epoch + 1, global_step, history)

    model.load_state_dict(best_state, strict=False)                  # restore best epoch
    history["best_f1"] = best_f1
    return history

In [13]:
@dataclass
class Experiment:
    name: str
    base_model: str               # one of PHASE1_CONFIGS
    method: str                   # full | partial | lora | lorma | qlora | adalora | adapter | prompt | prefix
    head_type: str                # mean | attn | bigru | chunk_max | label_attn
    strategy: str                 # head | tail | head_tail | uniform
    max_chunks: int               # chunks per case
    loss_type: str                # bce | weighted_bce | focal
    sampling: str                 # random | rare_oversample
    epochs: int
    lr: float                     # backbone / adapter / prefix learning rate
    head_lr: float                # head learning rate
    batch_docs: int
    grad_accum: int
    rank: int                     # LoRA / LoRMA / QLoRA / AdaLoRA rank
    alpha: float
    n_top: int                    # blocks trained by "partial"
    adapter_dim: int
    n_prefix: int                 # soft tokens for prompt / prefix tuning
    adalora_final_ratio: float    # fraction of AdaLoRA ranks kept after pruning
    block_stride: int             # 1 = all blocks; 2 = every 2nd block (KD student)
    teacher_name: str             # experiment whose exported logits are distilled (None = no KD)
    kd_temperature: float
    kd_hard_weight: float         # weight of the hard-label loss in the KD objective
    export_teacher_logits: bool   # save train-set logits so this model can teach a student
    run_section_probe: bool       # evaluate with only some sections of each case

BASE = Experiment(name="", base_model=MAIN_MODEL, method="lorma", head_type="mean", strategy="uniform",
                  max_chunks=DEFAULT_MAX_CHUNKS, loss_type="weighted_bce", sampling="random", epochs=8, lr=2e-4, head_lr=2e-3,
                  batch_docs=BATCH_DOCS, grad_accum=GRAD_ACCUM, rank=16, alpha=32.0, n_top=4, adapter_dim=64, n_prefix=16,
                  adalora_final_ratio=0.5, block_stride=1, teacher_name=None, kd_temperature=2.0, kd_hard_weight=0.5,
                  export_teacher_logits=False, run_section_probe=False)

def make_exp(name, **overrides):
    return replace(BASE, name=name, **overrides)

# ── teacher logits (for KD) ──
def teacher_logits_path(teacher_name):
    return f"{OUT}/teacher_logits_base_{teacher_name}.pt"

def save_teacher_logits(model, exp):
    logits = predict_logits(model, CaseDataset(train_split, exp.strategy, exp.max_chunks))
    torch.save({"ids": train_split.ids, "logits": logits, "strategy": exp.strategy, "max_chunks": exp.max_chunks},
               teacher_logits_path(exp.name))

def load_teacher_logits(exp):
    saved = torch.load(teacher_logits_path(exp.teacher_name), weights_only=False)
    assert saved["ids"] == train_split.ids, "teacher logits were exported for a different train set"
    assert saved["strategy"] == exp.strategy and saved["max_chunks"] == exp.max_chunks, "student must see the same chunks as the teacher"
    return saved["logits"]

# ── saved predictions / calibration (lets you re-threshold or ensemble later without retraining) ──
def load_calibration(name):
    saved = np.load(f"{OUT}/calib_{name}.npz")
    return {"temperature": float(saved["temperature"]), "global_t": float(saved["global_t"]), "per_label": saved["per_label"]}

def calibrated_probs(name, split_name):
    logits = np.load(f"{OUT}/{split_name}_logits_{name}.npy")
    return sigmoid(logits / load_calibration(name)["temperature"])

def thresholds_for_mode(calibration, mode):
    if mode == "global":
        return np.full(NUM_LABELS, calibration["global_t"])
    if mode == "per_label":
        return calibration["per_label"]
    raise ValueError(mode)

def write_submission(pred_matrix, name):
    submission = [{"id": case_id, "pred": sorted(LABEL_IDS[c] for c in np.where(row)[0])}
                  for case_id, row in zip(test_split.ids, pred_matrix)]
    valid_ids = {LABEL_IDS[c] for c in np.where(PREDICTABLE)[0]}
    assert [s["id"] for s in submission] == test_split.ids
    assert all(len(s["pred"]) > 0 and set(s["pred"]) <= valid_ids for s in submission)
    json_path, zip_path = f"{OUT}/submission_{name}.json", f"{OUT}/submission_{name}.zip"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, "prediction.json")
    print(f"  -> {zip_path}  (avg {np.mean([len(s['pred']) for s in submission]):.2f} statutes/case)")

def write_experiment_submission(name, mode):
    pred = decide(calibrated_probs(name, "test"), thresholds_for_mode(load_calibration(name), mode), PREDICTABLE)
    write_submission(pred, name)

PROBE_STRATEGIES = ["head", "tail", "head_tail", "uniform", "first_third", "middle_third", "last_third"]

def run_experiment(exp):
    """Build -> train -> calibrate -> tune thresholds -> write submission -> save results. Skips finished experiments."""
    result_path = f"{OUT}/result_{exp.name}.json"
    if os.path.exists(result_path):
        print(f"[{exp.name}] finished earlier -> skipped"); return

    set_seed(SEED)
    base_lm = load_phase1_model(exp.base_model)
    model = build_model(base_lm, exp)
    del base_lm; gc.collect()

    train_ds = CaseDataset(train_split, exp.strategy, exp.max_chunks)
    dev_ds   = CaseDataset(dev_split,   exp.strategy, exp.max_chunks)
    test_ds  = CaseDataset(test_split,  exp.strategy, exp.max_chunks)
    if exp.teacher_name is not None:
        train_ds.teacher_logits = load_teacher_logits(exp)

    total_m, trainable_m = count_params(model)
    print(f"\n=== {exp.name}: total {total_m:.1f}M | trainable {trainable_m:.2f}M ({100 * trainable_m / total_m:.1f}%)")
    torch.cuda.reset_peak_memory_stats()
    history = train_model(model, exp, train_ds, dev_ds)

    # dev inference (timed), calibration, thresholds, test inference
    torch.cuda.synchronize(); t0 = time.perf_counter()
    dev_logits = predict_logits(model, dev_ds).numpy()
    torch.cuda.synchronize(); infer_sec_per_case = (time.perf_counter() - t0) / len(dev_ds)
    peak_gb = torch.cuda.max_memory_allocated() / 2**30
    test_logits = predict_logits(model, test_ds).numpy()

    dev_labels = dev_split.labels
    temperature = fit_temperature(dev_logits, dev_labels)
    dev_probs = sigmoid(dev_logits / temperature)
    global_t, per_label_t = tune_thresholds(dev_probs, dev_labels)
    report = threshold_report(dev_probs, dev_labels, global_t, per_label_t)

    np.save(f"{OUT}/dev_logits_{exp.name}.npy", dev_logits); np.save(f"{OUT}/test_logits_{exp.name}.npy", test_logits)
    np.savez(f"{OUT}/calib_{exp.name}.npz", temperature=temperature, global_t=global_t, per_label=per_label_t)
    write_experiment_submission(exp.name, THRESHOLD_MODE)

    section_probe = None
    if exp.run_section_probe:                                        # which part of a case carries the signal? (fixed global threshold)
        section_probe = {}
        flat = np.full(NUM_LABELS, global_t)
        for strategy in PROBE_STRATEGIES:
            probe_logits = predict_logits(model, CaseDataset(dev_split, strategy, exp.max_chunks)).numpy()
            probe_pred = decide(sigmoid(probe_logits / temperature), flat, PREDICTABLE)
            section_probe[strategy] = multilabel_metrics(dev_labels, probe_pred)["F1_macro"]

    if exp.export_teacher_logits:
        save_teacher_logits(model, exp)

    result = {"name": exp.name, "config": exp.__dict__, "total_M": total_m, "trainable_M": trainable_m,
              "size_MB": model_size_mb(model), "temperature": temperature, "global_t": global_t, "dev": report,
              "train_min": history["train_sec"] / 60, "infer_ms_per_case": infer_sec_per_case * 1000,
              "peak_GB": peak_gb, "history": history, "section_probe": section_probe}
    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"### {exp.name}: trainable {trainable_m:.2f}M/{total_m:.1f}M | dev macro-F1 cross-fit: global {report['global_crossfit']['F1_macro']:.4f}, "
          f"per-label {report['per_label_crossfit']['F1_macro']:.4f} | in-sample global {report['global_insample']['F1_macro']:.4f} "
          f"| micro-F1 {report['global_crossfit']['F1_micro']:.4f} | subset acc {report['global_crossfit']['subset_acc']:.4f}")

    model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()

In [14]:
lm = load_phase1_model(MAIN_MODEL).to(DEVICE).eval()
x = torch.tensor([[BOS] + train_split.token_ids[0][:CONTENT_LEN]], device=DEVICE)
with torch.no_grad():
    logits = lm(x[:, :-1])
lm_loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1)).item()
print(f"Phase-1 LM loss on a train case: {lm_loss:.3f} (random init ~ {math.log(VOCAB_SIZE):.1f})")
del lm; gc.collect(); torch.cuda.empty_cache()

Phase-1 LM loss on a train case: 4.470 (random init ~ 9.9)


## 9. Smoke Test

In [14]:
# write_experiment_submission("base_full_finetune", "per_label")

  -> ./cs781_phase5/submission_base_full_finetune.zip  (avg 5.01 statutes/case)


In [15]:
if SMOKE_TEST:
    quick = dict(epochs=1, batch_docs=2, grad_accum=2)
    run_experiment(make_exp("smoke_full", method="full", export_teacher_logits=True, run_section_probe=True, **quick))
    run_experiment(make_exp("smoke_kd", method="full", block_stride=2, teacher_name="smoke_full", **quick))
    for method in ["partial", "lora", "lorma", "qlora", "adalora", "adapter", "prompt", "prefix"]:
        run_experiment(make_exp(f"smoke_{method}", method=method, **quick))
    for head_type in HEADS:
        run_experiment(make_exp(f"smoke_head_{head_type}", head_type=head_type, **quick))
    for loss_type in ["bce", "focal"]:
        run_experiment(make_exp(f"smoke_loss_{loss_type}", loss_type=loss_type, **quick))
    run_experiment(make_exp("smoke_oversample", sampling="rare_oversample", **quick))

## 10. Experiments

In [27]:
run_experiment(make_exp("base_full_finetune", method="full", epochs=2, lr=5e-5, head_lr=2e-3,
                        n_top=4, export_teacher_logits=True))


=== base_full_finetune: total 190.1M | trainable 190.10M (100.0%)
[base_full_finetune] resumed after epoch 2/2 (best so far 0.3790 at epoch 2)


In [31]:
# Teacher: full_finetune (12 blocks). Student: same Phase 1 model keeping every 2nd block (6 blocks), distilled from the teacher's per-statute logits.
run_experiment(make_exp("kd_student_half_depth", method="full", block_stride=2, teacher_name="full_finetune",
                        epochs=1, lr=5e-5, head_lr=1e-3, batch_docs=2, grad_accum=8,
                        kd_temperature=2.0, kd_hard_weight=0.5))


=== kd_student_half_depth: total 102.8M | trainable 102.77M (100.0%)


[kd_student_half_depth] ep 1/1 loss 0.9155 | dev macro P 0.338 R 0.411 F1 0.3641 | micro-F1 0.4643 @thr 0.50


  -> ./cs781_phase5/submission_kd_student_half_depth.zip  (avg 4.46 statutes/case)
### kd_student_half_depth: trainable 102.77M/102.8M | dev macro-F1 cross-fit: global 0.3625, per-label 0.3659 | in-sample global 0.3641 | micro-F1 0.4635 | subset acc 0.0342


In [ ]:
run_experiment(make_exp("lora_r16", method="lora", epochs=1, base_model= "gqa-rope", lr=5e-5, head_lr=2e-3,
                        n_top=4))


=== lora_r16: total 66.5M | trainable 0.73M (1.1%)


[lora_r16] ep 1/1 loss 0.3449 | dev macro P 0.226 R 0.375 F1 0.2739 | micro-F1 0.3869 @thr 0.42


  -> ./cs781_phase5/submission_lora_r16.zip  (avg 6.23 statutes/case)


predict:  13%|█▎        | 159/1255 [00:29<03:13,  5.66it/s] 

In [16]:
run_experiment(make_exp("lora_r16", method="lora", epochs=1, base_model= "gqa-rope", lr=5e-5, head_lr=2e-3,
                        n_top=4))


=== lora_r16: total 66.5M | trainable 0.73M (1.1%)
[lora_r16] resumed after epoch 1/1 (best so far 0.2739 at epoch 1)


  -> ./cs781_phase5/submission_lora_r16.zip  (avg 6.23 statutes/case)
### lora_r16: trainable 0.73M/66.5M | dev macro-F1 cross-fit: global 0.2733, per-label 0.2816 | in-sample global 0.2742 | micro-F1 0.3845 | subset acc 0.0109


In [17]:
run_experiment(make_exp("adapter_64", method="adapter", epochs=1, base_model="mla-sin", lr=5e-5, head_lr=2e-3,
                        n_top=4))


=== adapter_64: total 92.8M | trainable 1.27M (1.4%)


[adapter_64] ep 1/1 loss 0.3571 | dev macro P 0.194 R 0.371 F1 0.2471 | micro-F1 0.3629 @thr 0.38


  -> ./cs781_phase5/submission_adapter_64.zip  (avg 6.36 statutes/case)
### adapter_64: trainable 1.27M/92.8M | dev macro-F1 cross-fit: global 0.2467, per-label 0.2575 | in-sample global 0.2471 | micro-F1 0.3649 | subset acc 0.0073


In [ ]:
run_experiment(make_exp("lorma_r16", method="adapter", epochs=1, base_model="swiglu-moe-rope-conv", lr=5e-5, head_lr=2e-3,
                        n_top=4))


=== lorma_r16: total 198.6M | trainable 1.27M (0.6%)


[lorma_r16] ep 1/1 loss 0.3482 | dev macro P 0.214 R 0.379 F1 0.2657 | micro-F1 0.3775 @thr 0.40


predict:  76%|███████▌  | 948/1255 [07:34<01:51,  2.76it/s]